# Оценка текстов

In [ ]:
from pathlib import Path
import pandas as pd

data = pd.read_csv(Path.cwd().parent / "data/dataset_maxi_literal.csv")
data.info()

example_original_texts = data[data['text_domen'] == "художественный"][:5]
exmple_literal_texts =  data[data['text_domen'] == "новостной"][:5]

NameError: name 'Path' is not defined

## Оценка качества генерации (сохранение контекста)

**Используемые метрики:**
1. BERTScore -- cosine similarity between embeddings
2. BLEURT -- enhanced BERTScore (more alligned with human evalation)
3. COMET -- 
4. METEOR -- surface level matching (based on BLEU and ROUGE)

In [ ]:
from bert_score import score as bert_score_func
def measure_bertscore(references: list[str], candidates: list[str]) -> dict:
    """Вычисляет BERTScore (Precision, Recall, F1).
    
    Для русского языка используется xlm-roberta-base.
    rescale_with_baseline=True нормализует оценку, делая её более наглядной.
    """
    P, R, F1 = bert_score_func(
        cands=candidates, 
        refs=references, 
        lang="ru", 
        model_type="xlm-roberta-base",
        rescale_with_baseline=True
    )
    
    # Возвращаем средние значения по всему батчу текстов
    return {
        "bertscore_precision": P.mean().item(),
        "bertscore_recall": R.mean().item(),
        "bertscore_f1": F1.mean().item()
    }

In [ ]:
import os
from bleurt import score as bleurt_score
def measure_bluert(references: list[str], candidates: list[str], checkpoint_path: str = "./BLEURT-20") -> dict:
    """Вычисляет BLEURT score на основе мультиязычной модели BLEURT-20.
    
    Перед запуском нужно скачать архив модели BLEURT-20 и распаковать в рабочую директорию.
    """
    if not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"Чекпоинт BLEURT не найден по пути {checkpoint_path}. Скачайте и распакуйте BLEURT-20.")
        
    scorer = bleurt_score.BleurtScorer(checkpoint_path)
    scores = scorer.score(references=references, candidates=candidates)
    
    return {
        "bluert_score": sum(scores) / len(scores)
    }


In [ ]:
from comet import download_model, load_from_checkpoint
def measure_comet(references: list[str], candidates: list[str], sources: list[str] = None) -> dict:
    """Вычисляет COMET score (стандарт wmt22-comet-da).
    
    Если исходных текстов (sources) нет, передаем вместо них references, 
    чтобы модель могла выполнить синтаксическую проверку.
    """
    # Модель скачается автоматически при первом запуске
    model_path = download_model("Unbabel/wmt22-comet-da")
    model = load_from_checkpoint(model_path)
    
    if sources is None:
        sources = references
        
    # Форматируем данные под требования библиотеки
    data = [
        {"src": src, "mt": cand, "ref": ref}
        for src, cand, ref in zip(sources, candidates, references)
    ]
    
    model_output = model.predict(data, batch_size=8, gpus=1)
    
    return {
        "comet_score": model_output.system_score
    }

In [ ]:
import nltk
from evaluate import load

# Необходимые ресурсы NLTK для токенизации и стемминга
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

def measure_meteor(references: list[str], candidates: list[str]) -> dict:
    """Вычисляет METEOR score через стандартный интерфейс Hugging Face Evaluate."""
    meteor_metric = load('meteor')
    results = meteor_metric.compute(predictions=candidates, references=references)
    
    return {
        "meteor_score": results["meteor"]
    }

In [ ]:
def measure_all_metrics(references: list, candidates: list, sources: list = None) -> dict:
    results = {}
    
    return results

## Оценка идиоматичности

**Используемые метрики:**

Perplexity
Surpisal

*Лексические*
1. sentiment
2. concreteness
3. Word Mover's Distance (WMD)
4. inverse of the Jaccard similarity or edit distance

*Синтаксические*
1. Tree Edit Distance (?)
2. Comparing the sequence of POS tags between the original and literal sentences.
3. Selectional Preference Violation (сочетаемость сущ + глагол)
4. Syntactic Complexity & Parse Tree Features (можно просто находить конструкции)

*Семантические*
1. semantic distance between the words/phrases in your original text and their counterparts in the literal versions using contextual embeddings.
2. возможно пословный классификатор метафор
3. Embedding-Based Concreteness Direction (State-of-the-Art)
4. ContrastWSD (ContrastWSD: Enhancing Metaphor Detection with Word Sense Disambiguation Following the Metaphor Identification Procedure)

*Дискурсивные*

*Прагматические*

## LLM as a judge

1. Естественность употребления идиом (от 1 до 10 или похожее).
2. Образность текста (низкая, высокая, средняя).
3. Можно попросить выделить идиомы из текста и проверить согласованность разметки на примерах датасета.